In [ ]:
# ============================================================
# THE AROUSAL INDEX (P0a) — clean reproducible pipeline
# Interpretable, calibrated, cross-dataset model of musical arousal
# Runs top-to-bottom. Loads pre-extracted features (no re-extraction).
# ============================================================
import pandas as pd, numpy as np, glob, os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.isotonic import IsotonicRegression
from scipy.stats import pearsonr, spearmanr

DEAM = '/kaggle/input/datasets/imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music'
ANN  = f'{DEAM}/DEAM_Annotations/annotations/annotations averaged per song/song_level'

# --- DEAM labels: load, clean headers, merge, map 1-9 -> 0-11 ---
d1 = pd.read_csv(f'{ANN}/static_annotations_averaged_songs_1_2000.csv')
d2 = pd.read_csv(f'{ANN}/static_annotations_averaged_songs_2000_2058.csv')
d1.columns = d1.columns.str.strip(); d2.columns = d2.columns.str.strip()
shared = [c for c in d1.columns if c in d2.columns]
deam = pd.concat([d1[shared], d2[shared]], ignore_index=True)
deam['target'] = (deam['arousal_mean'] - 1.0)/(9.0-1.0)*11.0
print(f"DEAM labels: {len(deam)} songs, arousal {deam['arousal_mean'].min():.1f}-{deam['arousal_mean'].max():.1f}")

In [ ]:
import librosa, warnings
warnings.filterwarnings('ignore')

# The six named, interpretable features. Identical recipe for every dataset.
# Frame-level series -> mean + std per song. 30s cap for tractable runtime.
def extract_six(path):
    try:
        y, sr = librosa.load(path, sr=22050, mono=True, duration=30)
    except Exception:
        return None
    if len(y) < sr: return None
    rms  = librosa.feature.rms(y=y)[0]                    # loudness
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]  # brightness
    flux = librosa.onset.onset_strength(y=y, sr=sr)          # attack density
    zcr  = librosa.feature.zero_crossing_rate(y)[0]          # noisiness
    flat = librosa.feature.spectral_flatness(y=y)[0]         # sharpness proxy
    hnr  = -10*np.log10(flat + 1e-9)                         # harmonicity proxy
    return {'loudness_mean':rms.mean(),  'loudness_std':rms.std(),
            'brightness_mean':cent.mean(),'brightness_std':cent.std(),
            'flux_mean':flux.mean(),      'flux_std':flux.std(),
            'zcr_mean':zcr.mean(),        'zcr_std':zcr.std(),
            'sharpness_mean':flat.mean(), 'sharpness_std':flat.std(),
            'hnr_mean':hnr.mean(),        'hnr_std':hnr.std()}

SIX = ['loudness_mean','loudness_std','brightness_mean','brightness_std','flux_mean',
       'flux_std','zcr_mean','zcr_std','sharpness_mean','sharpness_std','hnr_mean','hnr_std']
print("Feature extractor defined. Features:", len(SIX))

In [ ]:
# Load the saved feature tables (extracted once; see extract_six above for the recipe).
# If you uploaded the CSVs as a dataset, point these at them. Otherwise re-extract.
FEAT_CSV_DEAM  = glob.glob('/kaggle/input/**/deam_librosa_features.csv', recursive=True)
FEAT_CSV_PMEMO = glob.glob('/kaggle/input/**/pmemo_librosa_features.csv', recursive=True)

if FEAT_CSV_DEAM and FEAT_CSV_PMEMO:
    deam_feat  = pd.read_csv(FEAT_CSV_DEAM[0])
    pmemo_feat = pd.read_csv(FEAT_CSV_PMEMO[0])
    print("Loaded pre-extracted features (fast path).")
else:
    print("CSVs not found as input — re-extracting (DEAM ~8min, PMEmo ~3min)...")
    DEAM_AUDIO = f'{DEAM}/DEAM_audio/MEMD_audio'
    PM_AUDIO   = '/kaggle/input/datasets/adityaraghuvanshi999/pmemo-original/chorus (2)/chorus'
    def build(folder, idcol):
        rows=[]
        for f in sorted(glob.glob(os.path.join(folder,'*.mp3'))):
            r=extract_six(f)
            if r: r[idcol]=int(os.path.basename(f).replace('.mp3','')); rows.append(r)
        return pd.DataFrame(rows)
    deam_feat  = build(DEAM_AUDIO,'song_id')
    pmemo_feat = build(PM_AUDIO,'musicId')

print(f"DEAM features: {deam_feat.shape}, PMEmo features: {pmemo_feat.shape}")

In [ ]:
deam_m = deam_feat.merge(deam[['song_id','target']], on='song_id')
cv = KFold(n_splits=5, shuffle=True, random_state=0)

# --- The PAPER's model: one representative per correlated cluster ---
# (spectral group brightness/zcr/sharpness/hnr inter-correlate |r|~0.6-0.9; see Cell 10 / Table 1.
#  loudness & flux are largely independent; flux excluded via the transfer ablation in Cell 13.)
CORE = ['zcr_mean','zcr_std','loudness_mean','loudness_std','hnr_mean','hnr_std']
assert not any('arousal' in c or 'target' in c for c in CORE), "LEAK!"

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
pred  = cross_val_predict(model, deam_m[CORE].values, deam_m['target'].values, cv=cv)
print(f"PAPER MODEL (decorrelated 3-cluster set), DEAM 5-fold:")
print(f"  R2 = {r2_score(deam_m['target'].values, pred):.3f}   "
      f"MAE = {mean_absolute_error(deam_m['target'].values, pred):.3f}")

model.fit(deam_m[CORE].values, deam_m['target'].values)
w = pd.Series(model.named_steps['ridge'].coef_, index=CORE).sort_values(key=abs, ascending=False)
print("\nInterpretable weights (every sign physically honest):")
print(w.round(3).to_string())

# --- Diagnostic only: full-12 model, to REPORT that collinearity flips signs ---
full = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
full_pred = cross_val_predict(full, deam_m[SIX].values, deam_m['target'].values, cv=cv)
print(f"\n(diagnostic) full-12 collinear model R2 = {r2_score(deam_m['target'].values, full_pred):.3f}"
      f" — similar accuracy, but signs flip (brightness hijacks zcr) => unusable for interpretation.")

In [ ]:
# Transfer test on the PAPER'S model (decorrelated CORE), not the full 12
pm_lab = pd.read_csv('/kaggle/input/datasets/adityaraghuvanshi999/pmemo-original/static_annotations.csv') \
           .rename(columns={'Arousal(mean)':'pm_arousal'})
pmemo_m = pmemo_feat.merge(pm_lab[['musicId','pm_arousal']], on='musicId')

# model is already fit on CORE from Cell 4-fixed
pm_pred = model.predict(pmemo_m[CORE].values)
pm_true = pmemo_m['pm_arousal'].values
sp = spearmanr(pm_true, pm_pred)[0]; pr = pearsonr(pm_true, pm_pred)[0]
print(f"PRIMARY MODEL transfer (decorrelated -> PMEmo, n={len(pm_true)}):")
print(f"  Spearman (headline) = {sp:.3f}")
print(f"  Pearson  = {pr:.3f}")

# For the paper's comparison table: full-12 transfer as the accuracy-reference
full.fit(deam_m[SIX].values, deam_m['target'].values)
fp = full.predict(pmemo_m[SIX].values)
print(f"\n(reference) full-12 transfer: Spearman = {spearmanr(pm_true, fp)[0]:.3f}, "
      f"Pearson = {pearsonr(pm_true, fp)[0]:.3f}")

In [ ]:
# ============================================================
# MODEL-SELECTION EXPERIMENT — resolves the invalid earlier comparison.
# Linear vs spline (GAM-style) vs boosted, ALL on the SAME decorrelated
# CORE features, judged on BOTH within-DEAM fit AND transfer to PMEmo.
# Lesson embodied: every number below is COMPUTED in this run — nothing
# is hardcoded.
# ============================================================
from sklearn.preprocessing import SplineTransformer
from sklearn.ensemble import GradientBoostingRegressor

Xc, yc = deam_m[CORE].values, deam_m['target'].values

candidates = {
    'Linear (Ridge)': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Spline / GAM':   make_pipeline(SplineTransformer(n_knots=4, degree=3),
                                    StandardScaler(with_mean=False), Ridge(alpha=10.0)),
    'Boosted (ref)':  GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0),
}

print(f"{'MODEL (same decorrelated 6)':<26}{'DEAM R2 (5-fold)':>18}{'PMEmo Spearman':>16}{'PMEmo Pearson':>15}")
print('-' * 75)
results = {}
for name, m in candidates.items():
    r2 = r2_score(yc, cross_val_predict(m, Xc, yc, cv=cv))
    m.fit(Xc, yc)
    p = m.predict(pmemo_m[CORE].values)
    results[name] = (r2, spearmanr(pm_true, p)[0], pearsonr(pm_true, p)[0])
    print(f"{name:<26}{results[name][0]:>18.3f}{results[name][1]:>16.3f}{results[name][2]:>15.3f}")
print('-' * 75)

lin_sp, spl_sp = results['Linear (Ridge)'][1], results['Spline / GAM'][1]
if spl_sp >= lin_sp + 0.02:
    print("READ: Spline transfers BETTER and fits better within-dataset ->")
    print("      spline/GAM should become the PRIMARY model (still interpretable:")
    print("      per-feature curves, just allowed to bend). Downstream cells")
    print("      (calibration, anchors, figure, tool) must switch to it.")
elif spl_sp >= lin_sp - 0.02:
    print("READ: Transfer is effectively TIED. Spline's within-dataset advantage")
    print("      (+~0.10 R2) then favors spline as primary, at a small readability")
    print("      cost (curved vs straight effects). Decide deliberately.")
else:
    print("READ: Spline fits better within-dataset but transfers WORSE -> keep")
    print("      LINEAR primary. Finding upgrades to: nonlinearity improves")
    print("      within-corpus fit but degrades cross-corpus transfer — more")
    print("      evidence that within-dataset accuracy rewards memorization.")


In [ ]:
# --- Isotonic calibration onto an honest 0-11 scale (v2: OUT-OF-FOLD) ---
calibrator = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=11) \
                 .fit(pred, deam_m['target'].values)          # pred = OOF from Cell 3
deam_m['calibrated'] = calibrator.predict(pred)               # honest scores for anchor picking
 
print("ANCHOR LADDER (v2, out-of-fold; DEAM lacks true 0-1 & 9-11 extremes):")
for point in range(2, 9):
    deam_m['d'] = (deam_m['calibrated']-point).abs() + (deam_m['target']-point).abs()
    b = deam_m.nsmallest(1,'d').iloc[0]
    print(f"  {point}/11 -> song {int(b['song_id']):4d} (model {b['calibrated']:.1f}, human {b['target']:.1f})")
print("\nPAPER CHECK (sec 4.4): confirm the 'well-populated 2-6, agreement ~0.3'")
print("and 'model ~6.5 vs human 8 at the top' claims against THIS output;")
print("adjust the two numbers in the tex if they moved.")
 

In [ ]:
import matplotlib.pyplot as plt
 
# --- Partial-dependence figure: how each driver moves arousal ---
SHOW = {'loudness_mean':'Loudness','zcr_mean':'Noisiness (ZCR)','hnr_mean':'Tonality index'}
means = deam_m[CORE].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax,(feat,lab) in zip(axes, SHOW.items()):
    grid = np.linspace(deam_m[feat].quantile(.02), deam_m[feat].quantile(.98), 100)
    Xt = pd.DataFrame([means]*100)[CORE].reset_index(drop=True); Xt[feat]=grid
    ax.plot(grid, model.predict(Xt.values), lw=2.5, color='#c0392b')
    ax.scatter(deam_m[feat], deam_m['target'], s=4, alpha=.12, color='gray')
    ax.set_xlabel(lab); ax.set_ylim(0,11)
axes[0].set_ylabel('Predicted arousal (0-11)')
fig.suptitle('The Arousal Index: interpretable feature effects', y=1.02)
plt.tight_layout(); plt.savefig('arousal_index_curves.png', dpi=150, bbox_inches='tight'); plt.show()
 
# --- The scoring tool ---
def score_arousal(path):
    f = extract_six(path)
    if f is None: return None
    raw_s = model.predict(pd.DataFrame([f])[CORE].values)[0]
    return round(float(calibrator.predict([raw_s])[0]), 1)
 
print("Pipeline complete. Tool ready: score_arousal('any.mp3')")
 

In [ ]:
# S1 — assemble the full corpus from 3 notebook tars + the B dataset
import subprocess, sys, os, glob, tarfile, shutil, collections, pandas as pd
subprocess.run([sys.executable,'-m','pip','install','-q','pyloudnorm'], check=True)
os.makedirs('corpus', exist_ok=True)
TAGS = {'mgs','mgm','sao','sa3s','ace'}

for t in glob.glob('/kaggle/input/**/audio_*.tar', recursive=True):
    with tarfile.open(t) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if b.endswith('.wav') and b.split('__')[0] in TAGS and not os.path.exists(f'corpus/{b}'):
                m.name = b; tf.extract(m, path='corpus')
for p in glob.glob('/kaggle/input/**/*.wav', recursive=True):
    b = os.path.basename(p)
    if b.split('__')[0] in TAGS and not os.path.exists(f'corpus/{b}'):
        shutil.copy(p, f'corpus/{b}')

wavs = sorted(glob.glob('corpus/*.wav'))
cnt = collections.Counter(os.path.basename(w).split('__')[0] for w in wavs)
print('corpus:', len(wavs), dict(cnt))   # want 1500: mgs 300, mgm 300, sao 300, sa3s 300, ace 300

pr = pd.read_csv(glob.glob('/kaggle/input/**/prompts.csv', recursive=True)[0])
rows = []
for w in wavs:
    mdl, pid, stail = os.path.basename(w)[:-4].split('__')
    rows.append(dict(file=w, model=mdl, prompt_id=pid, seed=int(stail[1:])))
M = pd.DataFrame(rows).merge(pr, on='prompt_id', how='left')
assert M.genre_slug.notna().all()
print(M.groupby(['model','phrasing']).size().unstack(fill_value=0))

In [ ]:
# S2 — LISTEN: calm vs intense, electronic-dance, per model
from IPython.display import Audio, display, HTML
for mdl in ['mgs','mgm','sao','sa3s','ace']:
    for pid in ['edm_L1_verb','edm_L9_verb']:
        f = f'corpus/{mdl}__{pid}__s0.wav'
        if os.path.exists(f):
            display(HTML(f'<b>{mdl} — {pid}</b>')); display(Audio(f))

In [ ]:
# S3 — score everything: p0a + LUFS + CLAP, plus -14 LUFS normalized rescoring (K4)
import numpy as np, librosa, torch, pyloudnorm as pyln, soundfile as sf, time
from transformers import ClapModel, ClapProcessor

clap  = ClapModel.from_pretrained('laion/clap-htsat-unfused').to('cuda').eval()
cproc = ClapProcessor.from_pretrained('laion/clap-htsat-unfused')
tf_ = cproc(text=['calm quiet gentle soft music','intense loud aggressive energetic music'],
            return_tensors='pt', padding=True).to('cuda')
def clap_delta(y48):
    af = cproc(audio=y48, sampling_rate=48000, return_tensors='pt').to('cuda')
    with torch.no_grad(): out = clap(**tf_, **af)
    te = out.text_embeds/out.text_embeds.norm(dim=-1, keepdim=True)
    ae = out.audio_embeds/out.audio_embeds.norm(dim=-1, keepdim=True)
    s = (ae @ te.T).squeeze(); return float(s[1]-s[0])

def score_all(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    lufs = pyln.Meter(sr).integrated_loudness(y)
    y48 = librosa.resample(y, orig_sr=sr, target_sr=48000)
    p0a, cd = score_arousal(path), clap_delta(y48)
    g = 10**((-14.0 - lufs)/20.0)
    sf.write('_tmp_norm.wav', np.clip(y*g, -1, 1), sr)
    return p0a, lufs, cd, score_arousal('_tmp_norm.wav'), clap_delta(np.clip(y48*g, -1, 1))

print("REAL-DEAM CONTROLS (instrument sanity):")
for _, r in pd.concat([deam_m.nsmallest(3,'target'), deam_m.nlargest(3,'target')]).iterrows():
    p0a, lufs, cd, _, _ = score_all(f"{DEAM}/DEAM_audio/MEMD_audio/{int(r['song_id'])}.mp3")
    print(f"  target={r['target']:.1f} -> p0a={p0a:.2f} lufs={lufs:.1f} clap={cd:.2f}")

SCORES = 'scores.csv'
done = set(pd.read_csv(SCORES)['file']) if os.path.exists(SCORES) else set()
todo = [f for f in M.file if f not in done]
print(len(todo), 'clips to score'); buf, t0 = [], time.time()
for i, f in enumerate(todo, 1):
    try:
        p0a, lufs, cd, p0a_n, cd_n = score_all(f)
        buf.append(dict(file=f, p0a=p0a, lufs=lufs, clap=cd, p0a_n=p0a_n, clap_n=cd_n))
    except Exception as e:
        buf.append(dict(file=f, p0a=np.nan, lufs=np.nan, clap=np.nan, p0a_n=np.nan, clap_n=np.nan))
        print('FAIL', os.path.basename(f), type(e).__name__)
    if i % 50 == 0 or i == len(todo):
        pd.DataFrame(buf).to_csv(SCORES, mode='a', header=not os.path.exists(SCORES), index=False)
        buf = []; r = i/max((time.time()-t0)/60, 0.01)
        print(f'{i}/{len(todo)} | {r:.0f} clips/min | ETA {(len(todo)-i)/r:.0f} min')
D = M.merge(pd.read_csv(SCORES), on='file'); print('scored:', len(D))

In [ ]:
# S4 — tables + compliance curves
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

print("MEAN P0A BY REQUESTED LEVEL:")
print(D.groupby(['model','phrasing','level']).p0a.mean().round(2).unstack())

d9 = D[D.level==9].groupby(['model','phrasing'])[['p0a','lufs','clap']].mean()
d1 = D[D.level==1].groupby(['model','phrasing'])[['p0a','lufs','clap']].mean()
print("\nΔ(L9 − L1):"); print((d9-d1).round(2))

rows = [dict(model=m, phrasing=ph, rho=round(spearmanr(g.level, g.p0a)[0], 3))
        for (m, ph), g in D.groupby(['model','phrasing'])]
print("\nSPEARMAN(level, p0a):")
print(pd.DataFrame(rows).pivot(index='model', columns='phrasing', values='rho'))

dn9 = D[D.level==9].groupby(['model','phrasing']).p0a_n.mean()
dn1 = D[D.level==1].groupby(['model','phrasing']).p0a_n.mean()
dec = pd.DataFrame({'raw_dp0a': (d9.p0a-d1.p0a).round(2), 'norm_dp0a': (dn9-dn1).round(2)})
dec['content_share'] = (dec.norm_dp0a/dec.raw_dp0a).round(2)
print("\nK4 VOLUME-KNOB DECOMPOSITION:"); print(dec)

rows = [dict(model=m, genre=gs, rho=spearmanr(g.level, g.p0a)[0])
        for (m, gs), g in D[D.phrasing=='verbal'].groupby(['model','genre_slug'])]
E = pd.DataFrame(rows).pivot(index='model', columns='genre', values='rho').round(3)
E['rock_minus_others'] = (E['rock'] - E[['edm','folk','hiphop','orchestral']].mean(axis=1)).round(3)
print("\nH1b ELASTICITY (verbal ρ by genre):"); print(E)

fig, axes = plt.subplots(1, 2, figsize=(12,4), sharey=True)
for ax, ph in zip(axes, ['numeric','verbal']):
    for m, g in D[D.phrasing==ph].groupby('model'):
        mm = g.groupby('level').p0a.mean(); ax.plot(mm.index, mm.values, marker='o', label=m)
    ax.set_title(f'{ph} prompts'); ax.set_xlabel('requested energy level'); ax.grid(alpha=.3)
axes[0].set_ylabel('P0A arousal (0-11)'); axes[0].legend()
plt.tight_layout(); plt.savefig('compliance_curves.png', dpi=150); plt.show()

In [ ]:
# S5 — bank results, discard the corpus copy (it lives in the source tars/dataset)
import shutil, os
D.to_csv('scored_full.csv', index=False)          # scores + all metadata, one portable file
shutil.rmtree('corpus', ignore_errors=True)
if os.path.exists('_tmp_norm.wav'): os.remove('_tmp_norm.wav')
print("BANKED: scores.csv, scored_full.csv, compliance_curves.png")
print("COMMIT COMPLETE — scoring day ran while you slept.")

In [ ]:
# ============================================================
# S6 (v2) — STATS: CIs, droop adjudication, phrasing contrast,
#      feature decomposition ("which knobs"), plausibility lens.
# Self-contained: re-loads scored_full.csv from disk (repairing the
# unquoted-comma malformation S5 writes: 13 header names / 14 fields),
# banks scored_full_v2.csv as the canonical analysis file, and never
# overwrites cells 0-7 names (`model`, `full`, `pred`, ...).
# Feature block (S6.4) needs cells 0-7 objects + audio; if S1-S5's live
# corpus/ was cleaned up, it REBUILDS the 750 verbal clips from the input
# tars (A/C/D) and loose input wavs (B) into /tmp, then deletes them after
# caching features. Degrades to a SKIP note if nothing is found.
# Every number below is COMPUTED in this run — nothing hardcoded.
# ============================================================
import os, glob, csv as _csv, time as _time
import numpy as np, pandas as pd
from scipy.stats import spearmanr as _spearmanr

S6_B    = 2000   # bootstrap resamples (pre-committed, P0B_SCOPE §6)
S6_SEED = 0      # every bootstrap call re-seeds -> outputs identical across
                 # runs/sections regardless of whether S6.4 executes
S6_LEVELS = [1, 3, 5, 7, 9]

# Q7 writability bands (CONSTRUCT_MAP v3.1, closed 2026-07-23; see P0B_HANDOFF §5).
# rock is an intensity-genre, not a Q7 palette: band below = H1 v2 catchy-zone
# (rock 5-8). ANALYST-SET — edit here if the variant band is preferred.
S6_BANDS = {'folk': (0.5, 5.5), 'edm': (4, 8), 'hiphop': (2, 7),
            'orchestral': (3, 7), 'rock': (5, 8)}

# ---------- S6.0 load + repair ----------
def _s6_load_scored_full():
    cands = ['scored_full.csv', '/kaggle/working/scored_full.csv'] + \
            sorted(glob.glob('/kaggle/input/**/scored_full.csv', recursive=True))
    path = next((p for p in cands if os.path.exists(p)), None)
    assert path, 'scored_full.csv not found (run S1-S5 first)'
    rows = [r for r in _csv.reader(open(path, newline='')) if r]
    widths = sorted({len(r) for r in rows[1:]})
    n14 = ['file','model','prompt_id','seed','genre_slug','level','phrasing',
           'genre_text','energy_text','p0a','lufs','clap','p0a_n','clap_n']
    if widths == [14]:      # S5's writer doesn't quote the comma in prompt_text
        d = pd.DataFrame(rows[1:], columns=n14)
        note = 'REPAIRED 14-field rows (unquoted comma in prompt_text)'
    elif widths == [13]:    # a fixed writer: prompt_text intact, split it
        d = pd.DataFrame(rows[1:], columns=rows[0])
        gt_et = d['prompt_text'].str.split(', ', n=1, expand=True)
        d['genre_text'], d['energy_text'] = gt_et[0], gt_et[1]
        d = d[n14]; note = 'clean 13-field file'
    else:
        raise ValueError(f'unexpected row widths {widths} in {path}')
    for c in ['p0a','lufs','clap','p0a_n','clap_n']: d[c] = pd.to_numeric(d[c])
    d['level'] = d['level'].astype(int); d['seed'] = d['seed'].astype(int)
    d['prompt_text'] = d['genre_text'] + ', ' + d['energy_text']
    d = d[['file','model','prompt_id','seed','genre_slug','level','phrasing',
           'genre_text','energy_text','prompt_text','p0a','lufs','clap','p0a_n','clap_n']]
    return d, path, note

S6, _s6_src, _s6_note = _s6_load_scored_full()
S6.to_csv('scored_full_v2.csv', index=False)   # properly quoted; canonical from here on
print(f'S6.0  loaded {len(S6)} rows from {_s6_src} ({_s6_note})')
print(f'      -> banked scored_full_v2.csv (quoted; 15 cols incl. genre_text/energy_text)')
assert len(S6) == 1500 and S6.file.is_unique, 'corpus shape drifted'

_SV = S6[S6.phrasing == 'verbal']
_SN = S6[S6.phrasing == 'numeric']

# ---------- bootstrap helpers (clip-level resampling throughout) ----------
def _ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def _boot_rho(lv, sc, B=S6_B):
    rng = np.random.default_rng(S6_SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([_spearmanr(lv[i], sc[i])[0] for i in idx])

def _boot_stat(arrs, fn, B=S6_B):
    """arrs: list of 1-D (or 2-D, clips x cols) arrays resampled independently
    (clip level, within group); fn maps the resampled list -> scalar."""
    rng = np.random.default_rng(S6_SEED)
    out = np.empty(B)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    for b in range(B):
        out[b] = fn([a[ix[b]] for a, ix in zip(arrs, idxs)])
    return out

_ci_rows = []   # long-format bank: analysis, model, scope, point, lo, hi, note
def _bank(analysis, mdl, scope, point, boots, note=''):
    lo, hi = _ci(boots)
    _ci_rows.append(dict(analysis=analysis, model=mdl, scope=scope,
                         point=round(float(point), 4), lo=round(float(lo), 4),
                         hi=round(float(hi), 4), note=note))
    return lo, hi

_MODELS = sorted(S6.model.unique())
_grp = {(m, ph): g for (m, ph), g in S6.groupby(['model', 'phrasing'])}
_lvl = {(m, L): g for (m, L), g in _SV.groupby(['model', 'level'])}

# ---------- S6.1 per-level means + L9 droop adjudication ----------
print('\n' + '='*72 + '\nS6.1  PER-LEVEL MEANS (droop adjudication)\n' + '='*72)
_lm = (S6.groupby(['model','phrasing','level'])[['p0a','clap','lufs','p0a_n','clap_n']]
         .mean().round(3))
_lm.to_csv('s6_level_means.csv')
for met in ['p0a', 'clap', 'lufs']:
    print(f'\nVERBAL mean {met} by level:')
    print(_lm.xs('verbal', level='phrasing')[met].unstack().round(2 if met!='clap' else 3).to_string())

print(f'\nDROOP PANEL — verbal L9 minus L7, 95% CI (clip-level, B={S6_B}):')
_droop = {}
for m in _MODELS:
    a7, a9 = _lvl[(m,7)], _lvl[(m,9)]
    for met in ['p0a', 'clap']:
        pt = a9[met].mean() - a7[met].mean()
        bo = _boot_stat([a7[met].values, a9[met].values], lambda xs: xs[1].mean()-xs[0].mean())
        lo, hi = _bank(f'droop_L9-L7_{met}', m, 'verbal', pt, bo)
        _droop[(m, met)] = (pt, lo, hi)
    dl = a9['lufs'].mean() - a7['lufs'].mean()
    print(f'  {m:5s} d_p0a {_droop[(m,"p0a")][0]:+.2f} [{_droop[(m,"p0a")][1]:+.2f},{_droop[(m,"p0a")][2]:+.2f}]'
          f'   d_clap {_droop[(m,"clap")][0]:+.3f} [{_droop[(m,"clap")][1]:+.3f},{_droop[(m,"clap")][2]:+.3f}]'
          f'   d_lufs {dl:+.2f} dB')
print('READ (own-prompt CLAP as second witness; raw-ridge check follows in S6.4):')
for m in _MODELS:
    p, plo, phi = _droop[(m,'p0a')]; c, clo, chi = _droop[(m,'clap')]
    if phi < 0 and chi < 0:
        v = 'both instruments drop -> MODEL BEHAVIOR (audio at L9 matches "extremely intense" less than L7 matched "energetic")'
    elif phi < 0 and clo > 0:
        v = 'p0a drops, CLAP rises -> suspect P0a top-end compression; adjudicate on raw ridge score (S6.4)'
    elif phi < 0:
        v = 'p0a drops, CLAP ~flat -> ambiguous; lean on raw ridge score (S6.4)'
    elif plo > 0:
        v = 'no droop (p0a rises L7->L9)'
    else:
        v = 'p0a droop CI includes 0 -> not established'
    print(f'  {m:5s} {v}')

# ---------- S6.2 monotonicity + dynamic-range CIs (+ K4 share CIs) ----------
print('\n' + '='*72 + '\nS6.2  BOOTSTRAP CIs — rho, deltas (mean & median), K4 share\n' + '='*72)
print(f'{"model":<6}{"phr":<9}{"rho [95% CI]":<26}{"d(L9-L1) mean [CI]":<26}{"median [CI]"}')
for m in _MODELS:
    for ph in ['numeric', 'verbal']:
        g = _grp[(m, ph)]
        rho = _spearmanr(g.level, g.p0a)[0]
        rb = _boot_rho(g.level.values, g.p0a.values)
        rlo, rhi = _bank('rho_level_p0a', m, ph, rho, rb)
        a1 = g[g.level==1].p0a.values; a9 = g[g.level==9].p0a.values
        dm = a9.mean()-a1.mean()
        db = _boot_stat([a1, a9], lambda xs: xs[1].mean()-xs[0].mean())
        dmlo, dmhi = _bank('delta_L9-L1_mean', m, ph, dm, db)
        dmd = np.median(a9)-np.median(a1)
        dbmd = _boot_stat([a1, a9], lambda xs: np.median(xs[1])-np.median(xs[0]))
        dmdlo, dmdhi = _bank('delta_L9-L1_median', m, ph, dmd, dbmd)
        star = '*' if dmlo > 0 else (' ' if dmhi >= 0 else 'v')
        print(f'{m:<6}{ph:<9}{rho:+.3f} [{rlo:+.3f},{rhi:+.3f}]   '
              f'{dm:+.2f} [{dmlo:+.2f},{dmhi:+.2f}]{star}  {dmd:+.2f} [{dmdlo:+.2f},{dmdhi:+.2f}]')
print("  '*' = mean-delta CI excludes 0.  NOTE: S4 headline used MEANS; P0B_SCOPE §6")
print('  pre-committed MEDIANS — where they diverge (skew = clip-concentrated failure),')
print('  report both in the draft and say which is pre-registered.')

print('\nK4 content_share = d_norm/d_raw (verbal, paired clip resample):')
for m in _MODELS:
    A1, A9 = _lvl[(m,1)][['p0a','p0a_n']].values, _lvl[(m,9)][['p0a','p0a_n']].values
    raw_d  = A9[:,0].mean()-A1[:,0].mean(); norm_d = A9[:,1].mean()-A1[:,1].mean()
    def _share(xs):
        r = xs[1][:,0].mean()-xs[0][:,0].mean()
        return (xs[1][:,1].mean()-xs[0][:,1].mean())/r if abs(r) > 1e-9 else np.nan
    sb = _boot_stat([A1, A9], _share)
    rb = _boot_stat([A1, A9], lambda xs: xs[1][:,0].mean()-xs[0][:,0].mean())
    unstable = (_ci(rb)[0] <= 0 <= _ci(rb)[1])
    note = 'UNSTABLE: raw-delta CI covers 0; ratio meaningless' if unstable else ''
    lo, hi = _bank('K4_content_share', m, 'verbal', norm_d/raw_d, sb, note)
    tail = f'  << {note}' if unstable else ''
    print(f'  {m:5s} share {norm_d/raw_d:+.2f} [{lo:+.2f},{hi:+.2f}]  (raw d {raw_d:+.2f}, norm d {norm_d:+.2f}){tail}')

# ---------- S6.3 phrasing contrast (verbal - numeric) ----------
print('\n' + '='*72 + '\nS6.3  PHRASING EFFECT — verbal minus numeric, per model\n' + '='*72)
for m in _MODELS:
    gn, gv = _grp[(m,'numeric')], _grp[(m,'verbal')]
    # rho contrast: independent clip resamples of each arm
    rvb = _boot_rho(gv.level.values, gv.p0a.values); rnb = _boot_rho(gn.level.values, gn.p0a.values)
    drho = _spearmanr(gv.level, gv.p0a)[0] - _spearmanr(gn.level, gn.p0a)[0]
    rlo, rhi = _bank('phrasing_drho', m, 'verbal-numeric', drho, rvb - rnb)
    arrs = [gn[gn.level==1].p0a.values, gn[gn.level==9].p0a.values,
            gv[gv.level==1].p0a.values, gv[gv.level==9].p0a.values]
    dd = (arrs[3].mean()-arrs[2].mean()) - (arrs[1].mean()-arrs[0].mean())
    ddb = _boot_stat(arrs, lambda xs: (xs[3].mean()-xs[2].mean())-(xs[1].mean()-xs[0].mean()))
    dlo, dhi = _bank('phrasing_ddelta', m, 'verbal-numeric', dd, ddb)
    sig = ' *' if rlo > 0 else ''
    print(f'  {m:5s} d_rho {drho:+.3f} [{rlo:+.3f},{rhi:+.3f}]{sig}   d_delta {dd:+.2f} [{dlo:+.2f},{dhi:+.2f}]')
print("  '*' = verbal beats numeric on monotonicity with CI excluding 0.")

# ---------- S6.4 feature block: raw ridge + which-knobs decomposition ----------
print('\n' + '='*72 + '\nS6.4  FEATURE DECOMPOSITION (audio-dependent; cached)\n' + '='*72)
_CLUSTERS = {'loudness': ['loudness_mean','loudness_std'],
             'noisiness': ['zcr_mean','zcr_std'],
             'tonality':  ['hnr_mean','hnr_std']}
try:
    _ = (model, extract_six, CORE, SIX)      # cells 0-7 objects, read-only
    _need = _SV.file.tolist()
    _cache_p = 'features_verbal.csv'
    _feat = pd.read_csv(_cache_p) if os.path.exists(_cache_p) else pd.DataFrame(columns=['file']+SIX)
    _feat = _feat[_feat.file.isin(_need)]
    _todo = [f for f in _need if f not in set(_feat.file)]
    if _todo:
        import tarfile, shutil
        _path_for, _S6C, _made_tmp = {}, '/tmp/s6_corpus', False
        # (1) live relative corpus from S1-S5, if it still exists
        for r in ['.', '/kaggle/working', '/tmp']:
            if os.path.exists(os.path.join(r, _todo[0])):
                for f in _todo:
                    p = os.path.join(r, f)
                    if os.path.exists(p): _path_for[f] = p
                print(f'  live corpus found under {r}/ ({len(_path_for)} clips)')
                break
        # (2) loose wavs anywhere in input/working/tmp (e.g. B dataset = loose sao)
        if len(_path_for) < len(_todo):
            _nb = {os.path.basename(f): f for f in _todo if f not in _path_for}
            for root in ['/kaggle/input', '/kaggle/working', '/tmp']:
                if not _nb: break
                for p in glob.glob(f'{root}/**/*.wav', recursive=True):
                    b = os.path.basename(p)
                    if b in _nb: _path_for[_nb.pop(b)] = p
        # (3) selective untar of input tars (A: mgs+mgm, C: sa3s, D: ace) -> /tmp
        _nb = {os.path.basename(f): f for f in _todo if f not in _path_for}
        if _nb:
            _tars = sorted(glob.glob('/kaggle/input/**/*.tar', recursive=True) +
                           glob.glob('/kaggle/input/**/*.tar.gz', recursive=True))
            if _tars: os.makedirs(_S6C, exist_ok=True); _made_tmp = True
            for t in _tars:
                if not _nb: break
                t1, hit = _time.time(), 0
                try:
                    with tarfile.open(t, 'r:*') as tf:
                        for mem in tf.getmembers():
                            b = os.path.basename(mem.name)
                            if mem.isfile() and b in _nb:
                                fo = tf.extractfile(mem)
                                if fo is None: continue
                                dst = os.path.join(_S6C, b)
                                with open(dst, 'wb') as out: out.write(fo.read())
                                _path_for[_nb.pop(b)] = dst; hit += 1
                    print(f'    {os.path.basename(t)}: pulled {hit} clips ({_time.time()-t1:.0f}s)')
                except Exception as te:
                    print(f'    tar skip {os.path.basename(t)}: {type(te).__name__}: {te}')
        print(f'  resolved audio for {len(_path_for)}/{len(_todo)} needed clips')
        if not _path_for:
            print(f'  SKIP extraction: no audio found for e.g. {_todo[0]}')
        else:
            print(f'  extracting {len(_path_for)} verbal clips (~0.3-0.6 s/clip) ...')
            t0, new = _time.time(), []
            for k, f in enumerate(_todo, 1):
                p = _path_for.get(f)
                if p is None: continue
                try: r = extract_six(p)
                except Exception: r = None
                if r: r['file'] = f; new.append(r)
                if k % 100 == 0: print(f'    {k}/{len(_todo)}  ({_time.time()-t0:.0f}s)')
            if new: _feat = pd.concat([_feat, pd.DataFrame(new)], ignore_index=True)
            for c6 in SIX: _feat[c6] = pd.to_numeric(_feat[c6])
            _feat.to_csv(_cache_p, index=False)
            print(f'  cached -> {_cache_p} ({len(_feat)}/{len(_need)} verbal clips)')
            if _made_tmp: shutil.rmtree(_S6C, ignore_errors=True)
    for c6 in SIX: _feat[c6] = pd.to_numeric(_feat[c6])
    cov = len(_feat)/len(_need)
    print(f'  coverage: {len(_feat)}/{len(_need)} verbal clips ({cov:.0%})')
    if cov < 0.90:
        print('  SKIP decomposition: coverage < 90%')
    else:
        _fm = _feat.merge(_SV[['file','model','level']], on='file')
        _sc, _rg = model.named_steps['standardscaler'], model.named_steps['ridge']
        Xz = (_fm[CORE].values - _sc.mean_) / _sc.scale_
        _fm['raw_score'] = Xz @ _rg.coef_ + _rg.intercept_
        _rawlvl = _fm.groupby(['model','level']).raw_score.mean().unstack()
        print('\n  RAW ridge score (pre-calibration), verbal means by level:')
        print(_rawlvl.round(2).to_string())
        _rawlvl.to_csv('s6_raw_ridge_level_means.csv')
        print('  READ (droop, second adjudicator):')
        for m in _MODELS:
            g7 = _fm[(_fm.model==m)&(_fm.level==7)].raw_score.values
            g9 = _fm[(_fm.model==m)&(_fm.level==9)].raw_score.values
            if not len(g7) or not len(g9): continue
            pt = g9.mean()-g7.mean()
            bo = _boot_stat([g7, g9], lambda xs: xs[1].mean()-xs[0].mean())
            lo, hi = _bank('droop_L9-L7_rawridge', m, 'verbal', pt, bo)
            v = ('raw score drops too -> droop is REAL model behavior, not calibration'
                 if hi < 0 else
                 ('raw score RISES -> calibrated droop = P0a top-end compression artifact'
                  if lo > 0 else 'raw droop CI includes 0 -> unresolved on this witness'))
            print(f'    {m:5s} d_raw(L9-L7) {pt:+.2f} [{lo:+.2f},{hi:+.2f}]  {v}')
        # which knobs: exact linear split of the raw-score delta, L9-L1
        print('\n  WHICH KNOBS — cluster contributions to raw-score delta (L9-L1, verbal):')
        print(f'  {"model":<6}{"d_raw":>7} | ' + ' | '.join(f'{c:>16}' for c in _CLUSTERS) + '  (contribution, share)')
        _dec = []
        for m in _MODELS:
            mu = _fm[_fm.model==m].groupby('level')[CORE].mean()
            if not {1,9} <= set(mu.index): continue
            dz = (mu.loc[9]-mu.loc[1]).values / _sc.scale_
            contrib = pd.Series(_rg.coef_ * dz, index=CORE)
            tot = contrib.sum()
            row = {'model': m, 'd_raw_total': round(tot,3)}
            cells = []
            for cname, feats in _CLUSTERS.items():
                cval = contrib[feats].sum()
                row[f'{cname}_contrib'] = round(cval,3)
                row[f'{cname}_share'] = round(cval/tot,3) if abs(tot) > 1e-9 else np.nan
                cells.append(f'{cval:+.2f} ({cval/tot:+.0%})' if abs(tot) > 1e-9 else f'{cval:+.2f} ( -- )')
            for f6 in SIX: row[f'dfeat_{f6}'] = (_fm[_fm.model==m].groupby("level")[f6].mean().loc[9]
                                                 - _fm[_fm.model==m].groupby("level")[f6].mean().loc[1])
            _dec.append(row)
            print(f'  {m:<6}{tot:+7.2f} | ' + ' | '.join(f'{c:>16}' for c in cells))
        pd.DataFrame(_dec).to_csv('s6_feature_decomp.csv', index=False)
        print('  (shares sum to 100% by construction; can exceed 100%/go negative when')
        print('   clusters oppose. Native-unit per-feature deltas banked in s6_feature_decomp.csv.)')
        print('  READ: loudness-cluster share ~ (1 - K4 content_share) is the consistency check;')
        print('        noisiness+tonality carrying the delta = "changed the music", per model.')
except NameError:
    print('  SKIP: cells 0-7 objects (model/extract_six/CORE) not in memory — Run All from top.')
except Exception as _e:
    print(f'  SKIP (crash-proofed): {type(_e).__name__}: {_e}')

# ---------- S6.5 plausibility lens: Q7 edge cells ----------
print('\n' + '='*72 + '\nS6.5  PLAUSIBILITY LENS — Q7 bands, edge cells (verbal)\n' + '='*72)
print('  bands: ' + ', '.join(f'{g} {lo}-{hi}' for g,(lo,hi) in S6_BANDS.items()) +
      '   (rock = H1 v2 catchy-zone; analyst-set)')
_ec = _SV.groupby(['model','genre_slug','level']).p0a.mean().round(2).unstack()
def _inb(g, L): lo, hi = S6_BANDS[g]; return lo <= L <= hi
_marked = _ec.copy().astype(object)
for g in _marked.index.get_level_values('genre_slug').unique():
    for L in S6_LEVELS:
        if not _inb(g, L):
            _marked.loc[(slice(None), g), L] = _marked.loc[(slice(None), g), L].map(lambda x: f'{x}*')
print('\n  mean p0a per cell ("*" = requested level OUTSIDE genre band):')
print(_marked.to_string())
_ec.to_csv('s6_edge_cells.csv')
print('\n  step-gain contrast: per-unit p0a gain on IN-band vs OUT/edge steps')
_steps = [(1,3),(3,5),(5,7),(7,9)]
for scope_m in _MODELS + ['ALL']:
    sub = _SV if scope_m == 'ALL' else _SV[_SV.model == scope_m]
    cells = {(g,L): gg.p0a.values for (g,L), gg in sub.groupby(['genre_slug','level'])}
    keys = sorted(cells); arrs = [cells[k] for k in keys]; pos = {k:i for i,k in enumerate(keys)}
    def _gap(xs, which):
        gains = []
        for g in S6_BANDS:
            for a,b in _steps:
                inb = _inb(g,a) and _inb(g,b)
                if inb == which and (g,a) in pos and (g,b) in pos:
                    gains.append((xs[pos[(g,b)]].mean()-xs[pos[(g,a)]].mean())/(b-a))
        return float(np.mean(gains))
    gin, gout = _gap(arrs, True), _gap(arrs, False)
    bo = _boot_stat(arrs, lambda xs: _gap(xs, True)-_gap(xs, False))
    lo, hi = _bank('edge_gain_in_minus_out', scope_m, 'verbal', gin-gout, bo)
    sig = ' *' if lo > 0 else ''
    print(f'  {scope_m:5s} in {gin:+.3f}  out {gout:+.3f}  diff {gin-gout:+.3f} [{lo:+.3f},{hi:+.3f}]{sig}')
print("  '*' = compliance concentrates INSIDE the plausible band (CI excludes 0).")

# ---------- S6.6 bank ----------
pd.DataFrame(_ci_rows).to_csv('s6_bootstrap_cis.csv', index=False)
print('\n' + '='*72 + '\nS6.6  BANKED: scored_full_v2.csv, s6_level_means.csv, s6_bootstrap_cis.csv,')
print('      s6_edge_cells.csv (+ s6_raw_ridge_level_means.csv, s6_feature_decomp.csv,')
print('      features_verbal.csv when audio present).  S6 complete.')

In [ ]:
# ============================================================
# S7 — review fixes: raw-scale CIs, sensitivity banks, leaderboard.
# NOTE: S7.1 (CLAP axis) is superseded by the standalone
# notebooks/clap-axis.ipynb; its skip here on any rerun is expected.
# Pays the two methodological debts from the red-team review and banks the
# sensitivity numbers already quoted in the draft:
#   S7.1 fixed-anchor CLAP axis for ALL 1,500 clips (deterministic: no
#        fusion, first-10s window) — replaces cross-level own-prompt
#        anchoring; fourth droop witness; text-side check for numeric.
#   S7.2 raw (pre-calibration) scale rho/delta CIs, verbal.
#   S7.3 plausibility contrast with the analyst-set rock band excluded.
#   S7.4 family contrasts with CIs (scale: mgm-mgs; era: sa3s-sao).
#   S7.5 clip-level L1/L9 scores bank (skew exhibit).
#   S7.6 leaderboard bank for README/IVE.
# Self-contained: reads scored_full_v2.csv (written by S6 this run) and the
# S6 feature cache; rebuilds audio from input tars only if the CLAP cache is
# incomplete. Every bootstrap re-seeds at 0 -> digit-identical across runs.
# The commit containing S6 v2 + S7 is CANONICAL-2: all paper numbers cite it.
# ============================================================
import os, glob, time as _t7
import numpy as np, pandas as pd
from scipy.stats import spearmanr as _sp7

S7_B, S7_SEED = 2000, 0
S7_CKPT = 'laion/larger_clap_music'   # HF transformers checkpoint for the
# axis (music-tuned, unfused -> deterministic). Falls back to
# 'laion/clap-htsat-unfused' if unavailable. Self-contained measurement:
# does not need to match whatever CLAP S1-S5 used for the own-prompt column.

S7 = pd.read_csv('scored_full_v2.csv')
assert len(S7) == 1500, 'run S6 first (scored_full_v2.csv missing/short)'
_SV7 = S7[S7.phrasing == 'verbal']
_M7 = sorted(S7.model.unique())

def _ci7(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def _brho7(lv, sc, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([_sp7(lv[i], sc[i])[0] for i in idx])

def _bstat7(arrs, fn, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    return np.array([fn([a[ix[b]] for a, ix in zip(arrs, idxs)]) for b in range(B)])

_rows7 = []
def _bank7(analysis, mdl, scope, point, boots, note=''):
    lo, hi = _ci7(boots)
    _rows7.append(dict(analysis=analysis, model=mdl, scope=scope,
                       point=round(float(point), 4), lo=round(float(lo), 4),
                       hi=round(float(hi), 4), note=note))
    return lo, hi

# ---------- S7.1 fixed-anchor CLAP axis ----------
print('='*72 + '\nS7.1  FIXED-ANCHOR CLAP AXIS (deterministic)\n' + '='*72)
_gt = S7.drop_duplicates('genre_slug').set_index('genre_slug')['genre_text'].to_dict()
_anch = {g: (f'{t}, very calm', f'{t}, extremely intense') for g, t in _gt.items()}
print('  anchors (genre-matched; differ only in the energy words):')
for g, (lo, hi) in sorted(_anch.items()): print(f'    {g}: "{lo}" | "{hi}"')

_axp = 's7_clap_axis.csv'
_ax = pd.read_csv(_axp) if os.path.exists(_axp) else pd.DataFrame(columns=['file','sim_lo','sim_hi','axis'])
_ax = _ax[_ax.file.isin(S7.file)]
_todo7 = [f for f in S7.file if f not in set(_ax.file)]
if _todo7:
    try:
        import tarfile, shutil, librosa as _lb7, torch
        # --- resolve audio (same 3-step logic as S6.4) ---
        _pf, _C7, _mk7 = {}, '/tmp/s7_corpus', False
        for r in ['.', '/kaggle/working', '/tmp']:
            if os.path.exists(os.path.join(r, _todo7[0])):
                for f in _todo7:
                    p = os.path.join(r, f)
                    if os.path.exists(p): _pf[f] = p
                break
        if len(_pf) < len(_todo7):
            _nb = {os.path.basename(f): f for f in _todo7 if f not in _pf}
            for root in ['/kaggle/input', '/kaggle/working', '/tmp']:
                if not _nb: break
                for p in glob.glob(f'{root}/**/*.wav', recursive=True):
                    b = os.path.basename(p)
                    if b in _nb: _pf[_nb.pop(b)] = p
        _nb = {os.path.basename(f): f for f in _todo7 if f not in _pf}
        if _nb:
            for t in sorted(glob.glob('/kaggle/input/**/*.tar', recursive=True) +
                            glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)):
                if not _nb: break
                try:
                    if not _mk7: os.makedirs(_C7, exist_ok=True); _mk7 = True
                    with tarfile.open(t, 'r:*') as tf:
                        n0 = len(_nb)
                        for mem in tf.getmembers():
                            b = os.path.basename(mem.name)
                            if mem.isfile() and b in _nb:
                                fo = tf.extractfile(mem)
                                if fo is None: continue
                                dst = os.path.join(_C7, b)
                                with open(dst, 'wb') as out: out.write(fo.read())
                                _pf[_nb.pop(b)] = dst
                        print(f'    {os.path.basename(t)}: pulled {n0-len(_nb)} clips')
                except Exception as te:
                    print(f'    tar skip {os.path.basename(t)}: {te}')
        print(f'  resolved audio for {len(_pf)}/{len(_todo7)} clips')
        assert _pf, 'no audio found'
        # --- deterministic CLAP via HF transformers (same stack as S1-S5) ---
        from transformers import ClapModel, ClapProcessor
        torch.manual_seed(0)
        _dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        try:
            _cm = ClapModel.from_pretrained(S7_CKPT).to(_dev).eval()
            _cp = ClapProcessor.from_pretrained(S7_CKPT); _used = S7_CKPT
        except Exception:
            _used = 'laion/clap-htsat-unfused'
            _cm = ClapModel.from_pretrained(_used).to(_dev).eval()
            _cp = ClapProcessor.from_pretrained(_used)
        print(f'  CLAP loaded via transformers: {_used} on {_dev}')
        _texts, _tkey = [], []
        for g, (a_lo, a_hi) in sorted(_anch.items()):
            _texts += [a_lo, a_hi]; _tkey += [(g,'lo'), (g,'hi')]
        def _proj(out, head):
            # transformers 4.x returns the projected tensor; 5.x returns the
            # encoder output object -> project pooler_output ourselves
            # (identical to the 4.x get_*_features implementation).
            if isinstance(out, torch.Tensor):
                t = out
            else:
                po = getattr(out, 'pooler_output', None)
                if po is None and isinstance(out, (tuple, list)) and len(out) > 1:
                    po = out[1]
                assert po is not None, f'no pooled output on {type(out).__name__}'
                t = head(po)
            return t.detach().cpu().numpy()
        _ti = _cp(text=_texts, return_tensors='pt', padding=True)
        with torch.no_grad():
            _te = _proj(_cm.get_text_features(**{k: v.to(_dev) for k, v in _ti.items()}),
                        _cm.text_projection)
        assert _te.ndim == 2 and _te.shape[0] == len(_texts), f'text emb shape {_te.shape}'
        _te = _te / np.linalg.norm(_te, axis=1, keepdims=True)
        _tix = {k: i for i, k in enumerate(_tkey)}
        _g_of = S7.set_index('file')['genre_slug'].to_dict()
        SRC, DUR = 48000, 10.0
        def _load10(p):
            y, _ = _lb7.load(p, sr=SRC, mono=True, duration=DUR)   # FIRST 10 s, deterministic
            out = np.zeros(int(SRC*DUR), dtype=np.float32); out[:len(y)] = y
            return out
        _new, _batchF, _batchY, t0 = [], [], [], _t7.time()
        def _flush():
            if not _batchF: return
            _ai = _cp(audios=[y for y in _batchY], sampling_rate=SRC,
                      return_tensors='pt', padding=True)
            with torch.no_grad():
                em = _proj(_cm.get_audio_features(**{k: v.to(_dev) for k, v in _ai.items()}),
                           _cm.audio_projection)
            assert em.shape[1] == _te.shape[1], \
                f'audio dim {em.shape[1]} != text dim {_te.shape[1]} (joint-space mismatch)'
            em = em / np.linalg.norm(em, axis=1, keepdims=True)
            for f, e in zip(_batchF, em):
                g = _g_of[f]
                s_lo = float(e @ _te[_tix[(g,'lo')]]); s_hi = float(e @ _te[_tix[(g,'hi')]])
                _new.append(dict(file=f, sim_lo=s_lo, sim_hi=s_hi, axis=s_hi-s_lo))
            _batchF.clear(); _batchY.clear()
        for k, f in enumerate(_todo7, 1):
            p = _pf.get(f)
            if p is None: continue
            try: _batchF.append(f); _batchY.append(_load10(p))
            except Exception: _batchF and _batchF.pop()
            if len(_batchF) == 16: _flush()
            if k == 4:                                  # smoke gate
                _flush()
                print('  SMOKE (first clips):')
                for r in _new[:4]: print(f"    {r['file']}: axis {r['axis']:+.4f}")
            if k % 200 == 0: print(f'    {k}/{len(_todo7)}  ({_t7.time()-t0:.0f}s)')
        _flush()
        _ax = pd.concat([_ax, pd.DataFrame(_new)], ignore_index=True)
        for c in ['sim_lo','sim_hi','axis']: _ax[c] = pd.to_numeric(_ax[c])
        _ax.to_csv(_axp, index=False)
        print(f'  cached -> {_axp} ({len(_ax)}/1500 clips)')
        if _mk7: shutil.rmtree(_C7, ignore_errors=True)
    except Exception as _e7:
        print(f'  SKIP CLAP axis (crash-proofed): {type(_e7).__name__}: {_e7}')

if len(_ax) >= 1350:            # >=90% coverage -> stats
    for c in ['sim_lo','sim_hi','axis']: _ax[c] = pd.to_numeric(_ax[c])
    _A = S7.merge(_ax, on='file')
    print('\n  axis means by level (verbal):')
    print(_A[_A.phrasing=='verbal'].groupby(['model','level']).axis.mean().unstack().round(3).to_string())
    _A.groupby(['model','phrasing','level'])['axis'].mean().round(4).to_csv('s7_axis_level_means.csv')
    print('\n  rho(level, axis) with 95% CI:')
    for m in _M7:
        for ph in ['numeric','verbal']:
            g = _A[(_A.model==m)&(_A.phrasing==ph)]
            r = _sp7(g.level, g.axis)[0]
            lo, hi = _bank7('rho_level_axis', m, ph, r, _brho7(g.level.values, g.axis.values))
            print(f'    {m:5s} {ph:8s} {r:+.3f} [{lo:+.3f},{hi:+.3f}]')
    print('  READ: axis is genre-controlled by construction (anchors share the')
    print('        genre phrase). Verbal rho >> numeric rho on the axis = the')
    print('        format effect confirmed on a synthetic-audio-native instrument.')
    print('\n  axis droop (verbal L9 - L7), fourth witness:')
    for m in _M7:
        g7 = _A[(_A.model==m)&(_A.phrasing=='verbal')&(_A.level==7)].axis.values
        g9 = _A[(_A.model==m)&(_A.phrasing=='verbal')&(_A.level==9)].axis.values
        pt = g9.mean()-g7.mean()
        lo, hi = _bank7('droop_L9-L7_axis', m, 'verbal', pt,
                        _bstat7([g7,g9], lambda xs: xs[1].mean()-xs[0].mean()))
        v = 'DROPS' if hi < 0 else ('rises' if lo > 0 else 'CI incl. 0')
        print(f'    {m:5s} {pt:+.4f} [{lo:+.4f},{hi:+.4f}]  {v}')
else:
    print(f'  axis stats skipped (coverage {len(_ax)}/1500)')

# ---------- S7.2 raw (pre-calibration) scale CIs, verbal ----------
print('\n' + '='*72 + '\nS7.2  RAW-SCALE CIs (verbal)\n' + '='*72)
try:
    _fe = pd.read_csv('features_verbal.csv')
    _sc7 = model.named_steps['standardscaler']; _rg7 = model.named_steps['ridge']
    _fm7 = _fe.merge(_SV7[['file','model','level']], on='file')
    _fm7['raw'] = ((_fm7[CORE].values - _sc7.mean_) / _sc7.scale_) @ _rg7.coef_ + _rg7.intercept_
    print(f'  {"model":<6}{"rho_raw [CI]":<24}{"d(9-1) mean [CI]":<24}{"median [CI]"}')
    for m in _M7:
        g = _fm7[_fm7.model==m]
        r = _sp7(g.level, g.raw)[0]
        rlo, rhi = _bank7('rho_level_raw', m, 'verbal', r, _brho7(g.level.values, g.raw.values))
        a1, a9 = g[g.level==1].raw.values, g[g.level==9].raw.values
        dm = a9.mean()-a1.mean()
        dlo, dhi = _bank7('delta_raw_mean', m, 'verbal', dm,
                          _bstat7([a1,a9], lambda xs: xs[1].mean()-xs[0].mean()))
        dmd = np.median(a9)-np.median(a1)
        mlo, mhi = _bank7('delta_raw_median', m, 'verbal', dmd,
                          _bstat7([a1,a9], lambda xs: np.median(xs[1])-np.median(xs[0])))
        print(f'  {m:<6}{r:+.3f} [{rlo:+.3f},{rhi:+.3f}]   {dm:+.2f} [{dlo:+.2f},{dhi:+.2f}]'
              f'    {dmd:+.2f} [{mlo:+.2f},{mhi:+.2f}]')
    print('  READ: raw-scale replicate of Table 1; ceiling-free ordering check.')
except Exception as _e7b:
    print(f'  SKIP (needs S6.4 feature cache + cells 0-7): {type(_e7b).__name__}: {_e7b}')

# ---------- S7.3 plausibility contrast, rock excluded ----------
print('\n' + '='*72 + '\nS7.3  BAND CONTRAST, ROCK EXCLUDED\n' + '='*72)
_B7 = {'folk':(0.5,5.5),'edm':(4,8),'hiphop':(2,7),'orchestral':(3,7),'rock':(5,8)}
_steps7 = [(1,3),(3,5),(5,7),(7,9)]
def _inb7(g,L): lo,hi=_B7[g]; return lo<=L<=hi
def _contrast7(sub, genres):
    cells = {(g,L):gg.p0a.values for (g,L),gg in sub.groupby(['genre_slug','level']) if g in genres}
    keys = sorted(cells); arrs=[cells[k] for k in keys]; pos={k:i for i,k in enumerate(keys)}
    def gap(xs, which):
        gains=[(xs[pos[(g,b)]].mean()-xs[pos[(g,a)]].mean())/(b-a)
               for g in genres for a,b in _steps7
               if (_inb7(g,a) and _inb7(g,b))==which and (g,a) in pos]
        return float(np.mean(gains))
    pt = gap(arrs,True)-gap(arrs,False)
    return pt, _bstat7(arrs, lambda xs: gap(xs,True)-gap(xs,False))
_ng = [g for g in _B7 if g != 'rock']
for scope in ['ALL'] + _M7:
    sub = _SV7 if scope=='ALL' else _SV7[_SV7.model==scope]
    pt, bo = _contrast7(sub, _ng)
    lo, hi = _bank7('edge_gain_norock', scope, 'verbal', pt, bo)
    print(f'  {scope:5s} in-out diff {pt:+.3f} [{lo:+.3f},{hi:+.3f}]{" *" if lo>0 else ""}')

# ---------- S7.4 family contrasts (scale, era) ----------
print('\n' + '='*72 + '\nS7.4  FAMILY CONTRASTS (verbal rho)\n' + '='*72)
def _rho_arm(m):
    g = _SV7[_SV7.model==m]; return g.level.values, g.p0a.values
for name, (m1, m2) in {'scale_mgm-mgs':('mgm','mgs'), 'era_sa3s-sao':('sa3s','sao')}.items():
    l1,s1 = _rho_arm(m1); l2,s2 = _rho_arm(m2)
    pt = _sp7(l1,s1)[0] - _sp7(l2,s2)[0]
    lo, hi = _bank7('family_drho', name, 'verbal', pt, _brho7(l1,s1)-_brho7(l2,s2))
    sig = 'CI excludes 0' if (lo>0 or hi<0) else 'CI includes 0 -> report as observation only'
    print(f'  {name}: d_rho {pt:+.3f} [{lo:+.3f},{hi:+.3f}]  ({sig})')

# ---------- S7.5 skew exhibit bank + S7.6 leaderboard ----------
_SV7[_SV7.level.isin([1,9])][['model','genre_slug','level','seed','p0a']] \
    .to_csv('s7_skew_L1L9.csv', index=False)
_lead = []
for m in _M7:
    g = _SV7[_SV7.model==m]; gn = S7[(S7.model==m)&(S7.phrasing=='numeric')]
    r = _sp7(g.level, g.p0a)[0]; rb = _brho7(g.level.values, g.p0a.values)
    a1,a9 = g[g.level==1].p0a.values, g[g.level==9].p0a.values
    A1 = g[g.level==1][['p0a','p0a_n']].values; A9 = g[g.level==9][['p0a','p0a_n']].values
    md = np.median(a9)-np.median(a1)
    mdb = _bstat7([a1,a9], lambda xs: np.median(xs[1])-np.median(xs[0]))
    raw_d = a9.mean()-a1.mean()
    rdb = _bstat7([a1,a9], lambda xs: xs[1].mean()-xs[0].mean())
    _unstable = _ci7(rdb)[0] <= 0 <= _ci7(rdb)[1]
    cs = np.nan if _unstable else (A9[:,1].mean()-A1[:,1].mean())/raw_d
    csb = _bstat7([A1,A9], lambda xs: (xs[1][:,1].mean()-xs[0][:,1].mean()) /
                  (xs[1][:,0].mean()-xs[0][:,0].mean()) if abs(xs[1][:,0].mean()-xs[0][:,0].mean())>1e-9 else np.nan)
    _lead.append(dict(model=m, house_level=round(gn.p0a.mean(),2),
                      rho_verbal=round(r,3), rho_lo=round(_ci7(rb)[0],3), rho_hi=round(_ci7(rb)[1],3),
                      med_delta=round(md,2), md_lo=round(_ci7(mdb)[0],2), md_hi=round(_ci7(mdb)[1],2),
                      content_share=round(cs,2) if not np.isnan(cs) else 'undefined',
                      cs_lo=round(_ci7(csb)[0],2), cs_hi=round(_ci7(csb)[1],2)))
pd.DataFrame(_lead).to_csv('s7_leaderboard.csv', index=False)
pd.DataFrame(_rows7).to_csv('s7_bootstrap_cis.csv', index=False)
print('\n' + '='*72 + '\nS7 BANKED: s7_clap_axis.csv, s7_axis_level_means.csv,')
print('  s7_bootstrap_cis.csv, s7_skew_L1L9.csv, s7_leaderboard.csv.  S7 complete.')